# Quality Checks: Gold tables

### Table : gold_dim_customers

In [ ]:
%%sql
-- After joining the tables, check if any duplicates were introuded by the join logic
SELECT cst_id, COUNT(*) FROM
(
SELECT
    ci.cst_id,
    ci.cst_key,
    ci.cst_firstname,
    ci.cst_lastname,
    ci.cst_marital_status,
    ci.cst_gndr,
    ci.cst_create_date,
    cu.bdate,
    cu.gen,
    la.cntry
FROM sales_lakehouse.dbo.silver_crm_customer_info ci
LEFT JOIN sales_lakehouse.dbo.silver_erp_customers cu
ON ci.cst_key = cu.cid
LEFT JOIN sales_lakehouse.dbo.silver_erp_location la
ON ci.cst_key = la.cid
)t
GROUP BY cst_id
HAVING COUNT(*) > 1;

-- observation: no results (no duplicates found)

In [ ]:
-- Integrity issue (we have two gender columns)

SELECT DISTINCT
    ci.cst_gndr,
    cu.gen
FROM sales_lakehouse.dbo.silver_crm_customer_info ci
LEFT JOIN sales_lakehouse.dbo.silver_erp_customers cu
ON ci.cst_key = cu.cid
LEFT JOIN sales_lakehouse.dbo.silver_erp_location la
ON ci.cst_key = la.cid
ORDER BY 1, 2;

-- observation: Different gender values coming from both the tables
-- solution: We need to fix any one table as the master table.
-- Consider CRM as the master table

In [ ]:
-- solution: We need to fix any one table as the master table.
-- Consider CRM as the master table
SELECT DISTINCT
    ci.cst_gndr,
    cu.gen,
    CASE WHEN ci.cst_gndr != 'Unknown' THEN ci.cst_gndr
         ELSE COALESCE(cu.gen, 'Unknown')
    END AS new_gen
FROM sales_lakehouse.dbo.silver_crm_customer_info ci
LEFT JOIN sales_lakehouse.dbo.silver_erp_customers cu
ON ci.cst_key = cu.cid
LEFT JOIN sales_lakehouse.dbo.silver_erp_location la
ON ci.cst_key = la.cid
ORDER BY 1, 2;

In [ ]:
-- check the low cardinary column after creating gold_dim_customer
SELECT DISTINCT gender FROM sales_lakehouse.dbo.gold_dim_customers;

### Table : gold_dim_products

In [ ]:
-- checking the quality of the result after joining
SELECT prd_key, COUNT(*) FROM
(SELECT
    pr.prd_id,
    pr.cat_id,
    pr.prd_key,
    pr.prd_nm,
    pr.prd_cost,
    pr.prd_line,
    pr.prd_start_dt,
    pc.cat,
    pc.subcat,
    pc.maintenance
FROM sales_lakehouse.dbo.silver_crm_product_info pr
LEFT JOIN sales_lakehouse.dbo.silver_erp_product_category pc
ON pr.cat_id = pc.id
WHERE pr.prd_end_dt IS NULL
)t
GROUP BY prd_key
HAVING COUNT(*) > 1;

-- observation: no duplicates found

In [ ]:
-- check if the data loaded
SELECT * FROM sales_lakehouse.dbo.gold_dim_products;

### Table: gold_fact_sales

In [ ]:
-- check if the data is loaded
SELECT * FROM sales_lakehouse.dbo.gold_fact_sales;

#### Foreign key integrity check (Dimensions) after creating gold fact table

In [ ]:
-- check if all dimension tables can successfully join to the fact table

SELECT * FROM sales_lakehouse.dbo.gold_fact_sales f
LEFT JOIN sales_lakehouse.dbo.gold_dim_customers c
ON f.customer_key = c.customer_key
WHERE c.customer_key IS NULL;

-- observation : no results (expected). All customer keys are matching

-- SELECT * FROM sales_lakehouse.dbo.gold_dim_customers LIMIT 1000

-- SELECT * FROM sales_lakehouse.dbo.gold_dim_products LIMIT 1000

In [ ]:
-- checking dim_product
SELECT * FROM sales_lakehouse.dbo.gold_fact_sales f
LEFT JOIN sales_lakehouse.dbo.gold_dim_products p
ON f.product_key = p.product_key
WHERE p.product_key IS NULL;

-- observation : no results (expected). All product keys are matching

#### Check after incremental load

In [ ]:
SELECT COUNT(*) FROM sales_lakehouse.dbo.silver_crm_sales_details;

In [ ]:
-- check no.of.records from gold_fact_sales
SELECT COUNT(*) FROM sales_lakehouse.dbo.gold_fact_sales;

In [ ]:
SELECT DISTINCT sls_ord_num FROM sales_lakehouse.dbo.silver_crm_sales_details s
LEFT JOIN sales_lakehouse.dbo.gold_fact_sales g
ON s.sls_ord_num != g.order_number
WHERE g.order_number IS NULL;

In [ ]:
SELECT DISTINCT sls_ord_num FROM sales_lakehouse.dbo.silver_crm_sales_details
EXCEPT
SELECT DISTINCT order_number FROM sales_lakehouse.dbo.gold_fact_sales;


In [ ]:
-- check for duplicate sls_ord_num
SELECT
sls_ord_num,
COUNT(*) as count_order
FROM sales_lakehouse.dbo.silver_crm_sales_details
GROUP BY sls_ord_num
HAVING COUNT(*) > 1;